# Phase 1 Baseline Early Fusion LSTM on CMU-MOSEI

Notebook này hỗ trợ profile `colab`, đọc dữ liệu từ Google Drive và lưu checkpoint/log/output lên Drive để dễ debug và tiếp tục train giữa các session.

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
else:
    print('Not running inside Google Colab. Mount step skipped.')

In [ ]:
import os
import sys
from pathlib import Path

RUNTIME_PROFILE = 'colab'
REPO_SOURCE = 'git'
REPO_URL = 'https://github.com/Kandesfx/Training-Multimodal-Emotion-Analysis.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/BCDA')
REPO_PATH = Path('/content/BCDA')
DATA_PKL_OVERRIDE = None
USE_DRIVE_OUTPUTS = True
RESUME_TRAINING = True
RESUME_CHECKPOINT_TYPE = 'last'
BEST_CHECKPOINT_NAME = 'best_model.pt'
LAST_CHECKPOINT_NAME = 'last_model.pt'
USE_GCS = True
GCS_BUCKET = 'mer-data-bucket-kandesfx'
WANDB_ENABLE = True
WANDB_PROJECT = 'bcda-phase1'

if RUNTIME_PROFILE != 'colab':
    raise ValueError('Notebook này hiện được tối ưu cho profile colab.')

if REPO_SOURCE not in {'git', 'drive'}:
    raise ValueError('REPO_SOURCE must be either \'git\' or \'drive\'.')

if REPO_SOURCE == 'git':
    if not REPO_PATH.exists():
        if '<YOUR_REPO_URL_HERE>' in REPO_URL:
            raise ValueError('Hãy thay REPO_URL bằng URL repo thật trước khi chạy cell này.')
        !git clone {REPO_URL} {REPO_PATH}
    else:
        print(f'Repo already exists at {REPO_PATH}')
else:
    REPO_PATH = DRIVE_ROOT

%cd {REPO_PATH}
if str(REPO_PATH) not in sys.path:
    sys.path.append(str(REPO_PATH))

!python -m pip install -q --upgrade pip
!python -m pip install -q torch torchvision torchaudio numpy pandas scikit-learn matplotlib seaborn tqdm wandb

if IN_COLAB and USE_GCS:
    from google.colab import auth
    print('Authenticating for GCS access...')
    auth.authenticate_user()
    print('Downloading aligned_50.pkl from GCS...')
    !mkdir -p /content/data/MSA-Dataset
    !gsutil cp gs://{GCS_BUCKET}/data/MSA-Dataset/aligned_50.pkl /content/data/MSA-Dataset/aligned_50.pkl


In [ ]:
from training.config_phase1 import config
from training.dataset_mosei import create_dataloaders
from training.models.early_fusion import EarlyFusionLSTMRegressor
from training.trainer import Phase1Trainer


In [ ]:
config.runtime.use_drive_outputs_on_colab = USE_DRIVE_OUTPUTS
config.runtime.use_gcs = USE_GCS
config.runtime.gcs_bucket = GCS_BUCKET
config.wandb.enable = WANDB_ENABLE
config.wandb.project = WANDB_PROJECT
config.apply_profile('colab', drive_root=DRIVE_ROOT, repo_root=REPO_PATH)

if DATA_PKL_OVERRIDE is not None:
    config.override_paths(mosei_pkl=DATA_PKL_OVERRIDE)

config.training.batch_size = 32
config.training.num_workers = 2
config.training.num_epochs = 10
config.training.learning_rate = 1e-3
config.training.use_amp = True
config.training.resume_from_checkpoint = RESUME_TRAINING
config.training.resume_checkpoint_type = RESUME_CHECKPOINT_TYPE
config.training.checkpoint_name = BEST_CHECKPOINT_NAME
config.training.last_checkpoint_name = LAST_CHECKPOINT_NAME
config.setup()

if USE_GCS and RESUME_TRAINING:
    print('Checking GCS for existing checkpoints to resume...')
    !mkdir -p {config.paths.checkpoints_dir}
    !gsutil cp gs://{GCS_BUCKET}/checkpoints/phase1/{BEST_CHECKPOINT_NAME} {config.paths.checkpoints_dir}/{BEST_CHECKPOINT_NAME} || true
    !gsutil cp gs://{GCS_BUCKET}/checkpoints/phase1/{LAST_CHECKPOINT_NAME} {config.paths.checkpoints_dir}/{LAST_CHECKPOINT_NAME} || true
    !mkdir -p {config.paths.logs_dir}
    !gsutil cp gs://{GCS_BUCKET}/logs/phase1/history.csv {config.paths.logs_dir}/history.csv || true

if WANDB_ENABLE:
    import wandb
    wandb.login()

print('Runtime profile:', config.runtime.profile)
print('Resume training:', config.training.resume_from_checkpoint)
print('Resume checkpoint type:', config.training.resume_checkpoint_type)
print('Best checkpoint name:', config.training.checkpoint_name)
print('Last checkpoint name:', config.training.last_checkpoint_name)
for key, value in config.paths.as_dict().items():
    print(f'{key}: {value}')


In [ ]:
required_paths = [
    config.paths.mosei_pkl,
    config.paths.checkpoints_dir,
    config.paths.logs_dir,
    config.paths.outputs_dir,
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required paths:\n' + '\n'.join(missing))

best_checkpoint_path = config.paths.checkpoints_dir / config.training.checkpoint_name
last_checkpoint_path = config.paths.checkpoints_dir / config.training.last_checkpoint_name

print(f'Best checkpoint path: {best_checkpoint_path}')
print(f'Last checkpoint path: {last_checkpoint_path}')
print(f'Best checkpoint exists: {best_checkpoint_path.exists()}')
print(f'Last checkpoint exists: {last_checkpoint_path.exists()}')
print('All required paths are ready.')

In [ ]:
dataloaders = create_dataloaders(config=config, pkl_path=config.paths.mosei_pkl)
sample_batch = next(iter(dataloaders['train']))

for key, value in sample_batch.items():
    if hasattr(value, 'shape'):
        print(key, value.shape, value.dtype)
    else:
        print(key, type(value), value[:2] if isinstance(value, list) else value)

In [ ]:
model = EarlyFusionLSTMRegressor(config.model)
trainer = Phase1Trainer(model=model, config=config)
summary = trainer.fit(dataloaders['train'], dataloaders['valid'])
test_metrics = trainer.evaluate_and_save(dataloaders['test'], split='test', epoch=summary['best_epoch'])
summary, test_metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_path = config.paths.logs_dir / 'history.csv'
history_df = pd.read_csv(history_path)
history_df.tail()

In [ ]:
valid_df = history_df[history_df['split'] == 'valid'].copy()
train_df = history_df[history_df['split'] == 'train'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_df['epoch'], train_df['loss'], label='train_loss')
if 'train_step_loss' in train_df.columns:
    axes[0].plot(train_df['epoch'], train_df['train_step_loss'], label='train_step_loss', linestyle='--')
axes[0].plot(valid_df['epoch'], valid_df['loss'], label='valid_loss')
axes[0].set_title('Loss curves')
axes[0].legend()

axes[1].plot(valid_df['epoch'], valid_df['mae'], label='valid_mae')
axes[1].plot(valid_df['epoch'], valid_df['corr'], label='valid_corr')
axes[1].set_title('Validation metrics')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print('Best checkpoint:', config.paths.checkpoints_dir / config.training.checkpoint_name)
print('Last checkpoint:', config.paths.checkpoints_dir / config.training.last_checkpoint_name)
print('Training summary:', config.paths.outputs_dir / 'summary.json')
print('History CSV:', config.paths.logs_dir / 'history.csv')